In [ ]:
import os
import requests
from dataclasses import dataclass, field
from typing import List, Optional
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent, ModelRetry, RunContext
from pydantic_ai.models.fallback import FallbackModel
from pydantic_ai.usage import UsageLimits

load_dotenv()

@dataclass
class WorkflowState:
    verified_papers: dict[str, dict] = field(default_factory=dict)  

class PaperRecommendation(BaseModel):
    found_match: bool = Field(description="True if a verified paper satisfied all constraints, False otherwise.")
    paper_title: Optional[str] = None
    authors: List[str] = Field(default_factory=list)
    publication_year: Optional[int] = None
    citation_count: Optional[int] = None
    citation_source: str = "OpenAlex"
    paper_link: Optional[str] = None
    relevance_explanation: str = Field(description="Why this paper matches the request, or why no paper matched.")
    uncertainty_notes: str = Field(description="Any caveats, e.g. missing abstract, approximate citations, or broad query.")

model = FallbackModel(
    "google:gemini-3.5-flash-lite",
    "google:gemini-3.1-flash-lite",
)

agent = Agent(
    model,
    deps_type=WorkflowState,
    output_type=PaperRecommendation,
    retries=2,
    system_prompt=(
        "You are an expert AI research assistant. Follow this workflow strictly:\n"
        "1. Parse the user's topic, year constraints, and citation constraints.\n"
        "2. Call `search_openalex_papers` with the extracted constraints.\n"
        "3. If no papers match, try 1-2 broader or alternative search queries.\n"
        "4. Reason about the relevance of the returned verified papers using their titles and abstracts.\n"
        "5. Recommend the single best matching paper using ONLY data returned by the tool.\n"
        "6. Always fill `uncertainty_notes` (e.g., if an abstract was missing, if citations are only from OpenAlex, "
        "or if no paper met all constraints—in which case set `found_match=False`)."
    ),
)

def reconstruct_abstract(inverted_index: Optional[dict]) -> str:
    """Deterministic helper: converts OpenAlex's inverted index into readable text."""
    if not inverted_index:
        return "No abstract available."
    words = [(pos, word) for word, positions in inverted_index.items() for pos in positions]
    words.sort(key=lambda x: x[0])
    return " ".join(word for _, word in words)[:500]


@agent.tool
def search_openalex_papers(
    ctx: RunContext[WorkflowState],
    query: str,
    min_year: Optional[int] = None,
    max_year: Optional[int] = None,
    exact_year: Optional[int] = None,
    min_citations: Optional[int] = None,
    max_citations: Optional[int] = None,
) -> list[dict]:
    """
    Searches OpenAlex for academic papers and deterministically filters them
    by publication year and citation count.
    """
    print(
        f"[Tool] Searching OpenAlex: query={query!r} | "
        f"year=(min={min_year}, max={max_year}, exact={exact_year}) | "
        f"citations=(min={min_citations}, max={max_citations})"
    )
    url = "https://api.openalex.org/works"

    filters = []
    if exact_year is not None:
        filters.append(f"publication_year:{exact_year}")
    else:
        if min_year is not None:
            filters.append(f"publication_year:>{min_year - 1}")
        if max_year is not None:
            filters.append(f"publication_year:<{max_year + 1}")
    if min_citations is not None:
        filters.append(f"cited_by_count:>{min_citations - 1}")
    if max_citations is not None:
        filters.append(f"cited_by_count:<{max_citations + 1}")

    params = {"search": query, "per-page": 10, "sort": "cited_by_count:desc"}
    if filters:
        params["filter"] = ",".join(filters)
    if os.getenv("OPENALEX_API_KEY"):
        params["api_key"] = os.getenv("OPENALEX_API_KEY")

    try:
        response = requests.get(url, params=params, timeout=15)
        print(f"[Tool] HTTP status: {response.status_code}")
        if response.status_code != 200:
            return [{"error": f"HTTP {response.status_code}: {response.text[:200]}"}]

        raw_results = response.json().get("results", [])
        verified = []
        rejected = 0

        
        for p in raw_results:
            year = p.get("publication_year")
            cites = p.get("cited_by_count")
            link = p.get("doi") or p.get("id")

            if exact_year is not None and year != exact_year:
                rejected += 1
                continue
            if min_year is not None and (year is None or year < min_year):
                rejected += 1
                continue
            if max_year is not None and (year is None or year > max_year):
                rejected += 1
                continue
            if min_citations is not None and (cites is None or cites < min_citations):
                rejected += 1
                continue
            if max_citations is not None and (cites is None or cites > max_citations):
                rejected += 1
                continue

            authors = [
                a["author"]["display_name"]
                for a in p.get("authorships", [])
                if a.get("author") and a["author"].get("display_name")
            ]
            paper_record = {
                "title": p.get("title"),
                "authors": authors[:5],
                "publication_year": year,
                "citation_count": cites,
                "citation_source": "OpenAlex",
                "url_or_doi": link,
                "abstract": reconstruct_abstract(p.get("abstract_inverted_index")),
            }
            verified.append(paper_record)
            if link:
                ctx.deps.verified_papers[link] = paper_record

        print(f"[Filter] Fetched {len(raw_results)} -> Kept {len(verified)}, Rejected {rejected}")
        return verified if verified else [{"message": "No papers matched these exact constraints."}]

    except requests.RequestException as exc:
        print(f"[Tool] Request failed: {exc}")
        return [{"error": f"Request failed: {exc}"}]


@agent.output_validator
def validate_recommendation(
    ctx: RunContext[WorkflowState], output: PaperRecommendation
) -> PaperRecommendation:
    print(f"[Validator] Checking output (found_match={output.found_match})...")
    if not output.found_match:
        print("[Validator] Passed (no matching paper found; uncertainty reported).")
        return output

    if not output.paper_link or output.paper_link not in ctx.deps.verified_papers:
        print("[Validator] REJECTED: Paper link was not in verified OpenAlex results.")
        raise ModelRetry(
            "The recommended paper_link was not returned by `search_openalex_papers`. "
            "You must select a paper from the verified tool results or set found_match=False."
        )

    verified = ctx.deps.verified_papers[output.paper_link]
    if output.citation_count != verified["citation_count"]:
        print("[Validator] REJECTED: Citation count mismatch.")
        raise ModelRetry(
            f"Citation count mismatch: tool returned {verified['citation_count']}, "
            f"not {output.citation_count}. Use the exact count from OpenAlex."
        )

    print("[Validator] Passed: Paper and citation count verified against OpenAlex.")
    return output


async def run_research_scout(prompt: str) -> PaperRecommendation:
    print(f"\n[Workflow] Starting request: {prompt!r}")
    state = WorkflowState()
    result = await agent.run(
        prompt,
        deps=state,
        usage_limits=UsageLimits(request_limit=8),
    )
    print("[Workflow] Finished.\n")
    return result.output


user_prompt = (
    "Find a research paper about LLM agents for software engineering that was "
    "published after 2022 and has at least 100 citations. Explain why the paper "
    "is relevant and provide the source of the citation count."
)

paper = await run_research_scout(user_prompt)

print(f"Found Match: {paper.found_match}")
print(f"Title: {paper.paper_title}")
print(f"Authors: {', '.join(paper.authors)}")
print(f"Year: {paper.publication_year}")
print(f"Citations: {paper.citation_count} (Source: {paper.citation_source})")
print(f"Link: {paper.paper_link}")
print(f"Explanation: {paper.relevance_explanation}")
print(f"Uncertainty: {paper.uncertainty_notes}")

In [ ]:
import asyncio
import json
from dataclasses import dataclass
from typing import Optional

@dataclass
class EvalCase:
    id: int
    category: str
    prompt: str
    expect_match: bool
    min_year: Optional[int] = None
    max_year: Optional[int] = None
    exact_year: Optional[int] = None
    min_citations: Optional[int] = None

EVAL_CASES = [
    EvalCase(
        id=1,
        category="Broad + After Year + Min Citations",
        prompt="Find a research paper about LLM agents for software engineering published after 2022 with at least 100 citations. Explain why it is relevant.",
        expect_match=True,
        min_year=2023,
        min_citations=100,
    ),
    EvalCase(
        id=2,
        category="Broad + Before Year + High Citations",
        prompt="Find a paper about retrieval-augmented generation published before 2021 with more than 500 citations. Summarize its contribution.",
        expect_match=True,
        max_year=2020,
        min_citations=501,
    ),
    EvalCase(
        id=3,
        category="Exact Year + Min Citations",
        prompt="Find a paper about transformer architectures published in 2017 with at least 1000 citations.",
        expect_match=True,
        exact_year=2017,
        min_citations=1000,
    ),
    EvalCase(
        id=4,
        category="Narrow Topic + After Year",
        prompt="Find a paper about multi-agent code generation or automated debugging published after 2023 with at least 20 citations.",
        expect_match=True,
        min_year=2024,
        min_citations=20,
    ),
    EvalCase(
        id=5,
        category="Narrow Topic + Exact Year",
        prompt="Find a paper about chain-of-thought prompting in large language models published in 2022 with at least 200 citations.",
        expect_match=True,
        exact_year=2022,
        min_citations=200,
    ),
    EvalCase(
        id=6,
        category="Broad + Before Year (Classic AI)",
        prompt="Find a paper about reinforcement learning published before 2016 with at least 500 citations.",
        expect_match=True,
        max_year=2015,
        min_citations=500,
    ),
    EvalCase(
        id=7,
        category="No Good Paper (Impossible Citations + Recent)",
        prompt="Find a paper about PydanticAI research agents published in 2026 with at least 50000 citations.",
        expect_match=False,
        exact_year=2026,
        min_citations=50000,
    ),
    EvalCase(
        id=8,
        category="No Good Paper (Anachronistic Topic)",
        prompt="Find a paper about GPT-4 tool calling published before 2015 with at least 50 citations.",
        expect_match=False,
        max_year=2014,
        min_citations=50,
    ),
    EvalCase(
        id=9,
        category="Ambiguous Request (No Explicit Numbers)",
        prompt="Find a recent paper about AI agents using tools and explain whether it would be useful for someone building autonomous software agents.",
        expect_match=True,
    ),
    EvalCase(
        id=10,
        category="Ambiguous Request (Vague Topic)",
        prompt="Find a really influential paper on making AI models safer or more aligned.",
        expect_match=True,
    ),
]

def check_constraints(case: EvalCase, paper: PaperRecommendation, state: WorkflowState) -> dict:
    """Deterministic checks for year, citations, valid source, and non-hallucination."""
    if not case.expect_match:
        correctly_abstained = not paper.found_match
        return {
            "year_ok": correctly_abstained,
            "citations_ok": correctly_abstained,
            "valid_source": True,
            "no_hallucination": correctly_abstained,
        }

    if not paper.found_match:
        return {
            "year_ok": False,
            "citations_ok": False,
            "valid_source": False,
            "no_hallucination": True,  
        }

    y = paper.publication_year
    year_ok = y is not None
    if year_ok and case.exact_year is not None:
        year_ok = (y == case.exact_year)
    if year_ok and case.min_year is not None:
        year_ok = (y >= case.min_year)
    if year_ok and case.max_year is not None:
        year_ok = (y <= case.max_year)

    c = paper.citation_count
    citations_ok = c is not None
    if citations_ok and case.min_citations is not None:
        citations_ok = (c >= case.min_citations)

    verified = state.verified_papers.get(paper.paper_link or "")
    valid_source = bool(paper.citation_source and paper.paper_link and verified is not None)
    no_hallucination = (
        verified is not None
        and paper.citation_count == verified["citation_count"]
        and paper.publication_year == verified["publication_year"]
    )

    return {
        "year_ok": year_ok,
        "citations_ok": citations_ok,
        "valid_source": valid_source,
        "no_hallucination": no_hallucination,
    }

async def run_evaluation():
    results = []
    for case in EVAL_CASES:
        print(f"\n================ TEST {case.id}: {case.category} ================")
        state = WorkflowState()
        try:
            res = await agent.run(
                case.prompt,
                deps=state,
                usage_limits=UsageLimits(request_limit=8),
            )
            paper = res.output
            checks = check_constraints(case, paper, state)
            record = {
                "id": case.id,
                "category": case.category,
                "found_match": paper.found_match,
                "title": paper.paper_title,
                "year": paper.publication_year,
                "citations": paper.citation_count,
                "link": paper.paper_link,
                "explanation": paper.relevance_explanation,
                "uncertainty": paper.uncertainty_notes,
                **checks,
            }
        except Exception as e:
            print(f"[ERROR] Test {case.id} failed with exception: {e}")
            record = {
                "id": case.id,
                "category": case.category,
                "error": str(e),
                "year_ok": False,
                "citations_ok": False,
                "valid_source": False,
                "no_hallucination": False,
            }

        results.append(record)

        await asyncio.sleep(4)

    with open("eval_results.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print("\n\n### Evaluation Summary Table\n")
    print("| ID | Category | Match | Year OK | Cite OK | Source OK | No Halluc. | Paper Title |")
    print("| :- | :--- | :-: | :-: | :-: | :-: | :-: | :--- |")
    for r in results:
        title = (r.get("title") or "None")[:45]
        print(
            f"| {r['id']} | {r['category']} | {r.get('found_match')} | "
            f"{'✅' if r['year_ok'] else '❌'} | {'✅' if r['citations_ok'] else '❌'} | "
            f"{'✅' if r['valid_source'] else '❌'} | {'✅' if r['no_hallucination'] else '❌'} | {title} |"
        )
    return results

eval_results = await run_evaluation()